In [ ]:
!git clone https://github.com/marcoshernanz/llm-lab.git /kaggle/working/llm-lab
%cd /kaggle/working/llm-lab
!git checkout milestone-027
!python -m pip install -q -U pip
!python -m pip install -q -U "jax[tpu]" flax numpy pyarrow datasets huggingface-hub


In [ ]:
"""Download the public tokenizer and shard files from Hugging Face."""

from pathlib import Path
from huggingface_hub import snapshot_download

local_data_root = Path("/kaggle/working/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384")
local_data_root.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id="marcoshernanz/llm-lab-fineweb-edu-sample10bt-bpe-16384",
    repo_type="dataset",
    local_dir=str(local_data_root),
    allow_patterns=["*.json", "*.npy"],
    token=False,
)

print("local_data_root=", local_data_root)
for path in sorted(local_data_root.iterdir()):
    print("  ", path.name)


In [ ]:
!PYTHONPATH=/kaggle/working/llm-lab python experiments/027_tpu_fineweb_edu_adam.py \
  --token-shard-root /kaggle/working/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384 \
  --tokenizer-path /kaggle/working/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384/fineweb_edu_sample10bt_bpe_16384.json \
  --artifacts-root /kaggle/working/artifacts/experiments \
  --execution-target "Kaggle TPU v5e-8 milestone-027 baseline" \
  --max-train-shards 10 \
  --train-steps 100000 \
  --train-chunk-length 100 \
  --batch-size 128 \
  --learning-rate 0.001 \
  --beta1 0.9 \
  --beta2 0.999 \
  --epsilon 1e-8


In [ ]:
"""Display the newest saved milestone-027 run."""

from IPython.display import SVG, display
import json
from pathlib import Path

run_dirs = sorted(Path("/kaggle/working/artifacts/experiments/027_tpu_fineweb_edu_adam").glob("*"))
if not run_dirs:
    raise FileNotFoundError("No milestone-027 artifact directory was created.")

latest_run_dir = run_dirs[-1]
print("latest_run_dir=", latest_run_dir)
display(SVG(filename=str(latest_run_dir / "loss_curve.svg")))
print(json.dumps(json.loads((latest_run_dir / "run_metadata.json").read_text(encoding="utf-8")), indent=2))
print((latest_run_dir / "sample.txt").read_text(encoding="utf-8"))


In [ ]:
"""Zip the newest milestone-027 run for download from Kaggle."""

from pathlib import Path
import shutil

runs_root = Path("/kaggle/working/artifacts/experiments/027_tpu_fineweb_edu_adam")
run_dirs = sorted(path for path in runs_root.glob("*") if path.is_dir())
if not run_dirs:
    raise FileNotFoundError(f"No runs found under {runs_root}")

latest_run_dir = run_dirs[-1]
archive_base = Path("/kaggle/working") / latest_run_dir.name
archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=str(latest_run_dir.parent),
        base_dir=latest_run_dir.name,
    )
)

print("latest_run_dir=", latest_run_dir)
print("archive_path=", archive_path)
print("download_from_kaggle_output_panel=", archive_path.name)
